# Проверка наших моделей YOLO12m на LinQ TPU (v3)

Версия без cv2 и onnxruntime: нужен только numpy + PIL + pika (всё уже есть на стенде).

**Загрузите в папку этого ноутбука (кнопка Upload) файлы:**
1. `person-yolo12m-head-b1-o1-h1m.tpu` — наша обученная модель детекции людей (основная)
2. `yolo12m-head-b1-o1-h1m.tpu` — стоковая COCO YOLO12m (запасная, можно не загружать)
3. `test-person.jpg` — тестовый кадр с человеком
4. `reference-yolo12m.npz` — эталонные FP32-выходы для сравнения (необязательно)

**В ячейке «Настройки» проверьте LOGIN.** Выполнять ячейки строго по порядку.
Программа автоматически копируется в /auto/cache/<LOGIN>/ (ячейка 2).
Decode/NMS выполняются на хосте чистым numpy (шаги сетки 8/16/32 — проверено локально до точного совидения с ONNX-хвостом).
Ячейка 8 — обязательный по кейсу замер времени; итог — файл linq-timing.json + скриншот ячейки для отчёта.

In [ ]:
# Read-only environment check. No automatic uninstall/delete in Run All.
import numpy as np
np.zeros(3)
print('numpy OK:', np.__version__, np.__file__)


In [ ]:
# 1. Настройки
import os

# Командный логин из integration kit; сохраняйте его для текущей программы.
LOGIN = 'hackathon74291'

# Основная программа: наша обученная модель людей.
# Альтернатива: 'yolo12m-head-b1-o1-h1m.tpu' (стоковая COCO YOLO12m)
PROGRAM = 'person-yolo12m-head-b1-o1-h1m.tpu'

IMG = 'test-person.jpg'                     # тестовый кадр
REFERENCE_NPZ = 'reference-person.npz'      # FP32-эталон person-модели (необязательно)

PROGRAM_DIR = os.path.join('/auto/cache', LOGIN)
PROGRAM_PATH = os.path.join(PROGRAM_DIR, PROGRAM)

# --- настройки брокера (из примера организаторов) ---
RABBITMQ_HOST = '10.0.114.30'
EXCHANGE = 'tpu.direct'
INPUT_QUEUE = 'input'
AMQP_PORT = 5672
CREDENTIALS = ('guest', 'guest')
TIMEOUT_SECONDS = 120

# --- контракт наших программ ---
INPUT_TENSOR_NAME = 'images'                # float32 (1,3,640,640), RGB /255 после letterbox
CUTS = ['/model.21/Div_1_output_0',
        '/model.21/Sub_1_output_0',
        '/model.21/Sigmoid_output_0']
INPUT_HW = (640, 640)
CONF_THRESH = 0.15   # обученная person-модель на тестовых кадрах даёт 0.2-0.55; 0.4 отсекал всё
IOU_THRESH = 0.45
print('настройки готовы, программа:', PROGRAM_PATH)

In [ ]:
# 2. Копируем программу в каталог сервиса
import shutil, os
os.makedirs(PROGRAM_DIR, exist_ok=True)
assert __import__('hashlib').sha256(open(PROGRAM,'rb').read()).hexdigest() == '37281bff05117d546f1bb76d4431c41b8e7e00a88c46a16578e46e19b918cc95', 'Unexpected TPU program'
shutil.copy(PROGRAM, PROGRAM_PATH)
print('программа скопирована:', PROGRAM_PATH, os.path.getsize(PROGRAM_PATH), 'байт')

In [ ]:
# 3. Препроцессинг: letterbox 640x640, RGB, /255, NCHW (как в проекте)
from PIL import Image

def letterbox(img_rgb, size=INPUT_HW):
    w, h = img_rgb.size; tw, th = size; r = min(tw / w, th / h)
    nw, nh = round(w * r), round(h * r)
    left, top = round((tw - nw) / 2 - .1), round((th - nh) / 2 - .1)
    padded = Image.new('RGB', (tw, th), (114, 114, 114))
    padded.paste(img_rgb.resize((nw, nh), Image.Resampling.BILINEAR), (left, top))
    arr = np.asarray(padded, dtype=np.float32).transpose(2, 0, 1)[None] / 255
    return np.ascontiguousarray(arr), (r, left, top)

image = Image.open(IMG).convert('RGB')
assert image is not None
tensor, (SCALE, PAD_L, PAD_T) = letterbox(image)
print('вход:', tensor.shape, tensor.dtype, 'min=%.3f max=%.3f' % (tensor.min(), tensor.max()))

In [ ]:
# 4. Отправляем тензор в сервис LinQ (как в примере организаторов)
import json, time, uuid
import pika
from tpu_pycloud_format import (MESSAGE_ID_ATTR_NAME, RESPONSE_ROUTING_KEYS_ATTR_NAME,
                                TARGET_URI_ATTR_NAME, PyTask, serialize)

task_id = uuid.uuid4().hex
reply_queue = f'notebook_{task_id[:8]}'

connection = pika.BlockingConnection(pika.ConnectionParameters(
    RABBITMQ_HOST, AMQP_PORT, credentials=pika.PlainCredentials(*CREDENTIALS)))
channel = connection.channel()
channel.queue_declare(queue=reply_queue, auto_delete=True)
channel.queue_bind(exchange=EXCHANGE, queue=reply_queue, routing_key=reply_queue)

tensors = {INPUT_TENSOR_NAME: tensor}
started = time.monotonic()
channel.basic_publish(
    exchange=EXCHANGE,
    routing_key=INPUT_QUEUE,
    body=serialize(PyTask(tensors)),
    properties=pika.BasicProperties(headers={
        TARGET_URI_ATTR_NAME: f'file://{PROGRAM_PATH}?v=2&raw=False',
        RESPONSE_ROUTING_KEYS_ATTR_NAME: json.dumps([reply_queue]),
        MESSAGE_ID_ATTR_NAME: task_id,
    }),
)
print(f'отправлено: задача {task_id}, ответ ждём в {reply_queue}')

In [ ]:
# 5. Ждём ответ воркера
from tpu_pycloud_format import deserialize

for method, properties, body in channel.consume(reply_queue, auto_ack=True, inactivity_timeout=1.0):
    if method is None:
        if time.monotonic() - started > TIMEOUT_SECONDS:
            raise TimeoutError(f'нет ответа за {TIMEOUT_SECONDS} с')
        continue
    channel.cancel()
    break
connection.close()

assert properties.headers[MESSAGE_ID_ATTR_NAME] == task_id, 'ответ не на нашу задачу'
data, metadata, worker = deserialize(body)
elapsed = time.monotonic() - started
print(f'ответ за {elapsed:.2f} с от воркера pid {worker["worker_pid"]}, status_code={worker["status_code"]}')

if not data:
    raise RuntimeError(f'ошибка воркера: {worker.get("error_message")}')
print('ключи ответа:', list(data.keys()))
tpu_out = {name: np.asarray(data[name], dtype=np.float32) for name in CUTS if name in data}
for name, arr in tpu_out.items():
    print(f'{name}: shape={arr.shape} dtype={arr.dtype} min={arr.min():.4f} max={arr.max():.4f}')

In [ ]:
# 6. Decode + NMS чистым numpy: боксы умножаются на шаги сетки (8/16/32),
#    затем конкатенируются с сигмоидами классов — это ровно то, что делал tail-граф
#    (проверено локально: расхождение с ONNX-хвостом = 0.0)
STRIDES = np.repeat([8., 16., 32.], [6400, 1600, 400])[None].astype(np.float32)

div = tpu_out[CUTS[0]]   # (1,2,8400)
sub = tpu_out[CUTS[1]]   # (1,2,8400)
sig = tpu_out[CUTS[2]]   # (1,80,8400)
out = np.concatenate([np.concatenate([div, sub], 1) * STRIDES, sig], 1)[0].T  # (8400,84)

boxes, scores = out[:, :4], out[:, 4:]
cls = scores.argmax(1); conf = scores[np.arange(len(out)), cls]
cand = sorted((i for i in range(len(out)) if cls[i] == 0 and conf[i] >= CONF_THRESH),
              key=lambda i: -conf[i])
picked = []
for i in cand:
    x, y, w, h = boxes[i]
    bx = np.array([(x - w/2 - PAD_L)/SCALE, (y - h/2 - PAD_T)/SCALE,
                   (x + w/2 - PAD_L)/SCALE, (y + h/2 - PAD_T)/SCALE])
    ok = True
    for p, _ in picked:
        ix = max(0, min(bx[2], p[2]) - max(bx[0], p[0])); iy = max(0, min(bx[3], p[3]) - max(bx[1], p[1]))
        inter = ix * iy; a = (bx[2]-bx[0])*(bx[3]-bx[1]); b = (p[2]-p[0])*(p[3]-p[1])
        if inter / max(a + b - inter, 1e-9) > IOU_THRESH: ok = False; break
    if ok: picked.append((bx, conf[i]))

print(f'детекций человека (conf>={CONF_THRESH}): {len(picked)}')
from PIL import ImageDraw
annotated = image.copy()
drw = ImageDraw.Draw(annotated)
for bx, c in picked:
    drw.rectangle([float(bx[0]), float(bx[1]), float(bx[2]), float(bx[3])], outline=(0, 255, 0), width=2)
    drw.text((float(bx[0]), max(0.0, float(bx[1]) - 10)), 'person %.2f' % c, fill=(0, 255, 0))
    print('  person conf=%.3f box=%s' % (c, np.round(bx).astype(int)))
annotated.save('tpu-detections.jpg')
display(annotated)

In [ ]:
# 7. Сравнение с локальным FP32-эталоном.
# INT8-квантование даёт численное расхождение — это ожидаемо. Важны высокая корреляция
# и совпадение итоговых детекций, а не нулевая ошибка.
import os
if not os.path.exists(REFERENCE_NPZ):
    print('эталон не загружен, пропускаем сравнение')
else:
    ref = np.load(REFERENCE_NPZ)
    pairs = [(CUTS[0], ref['ref_div']), (CUTS[1], ref['ref_sub']), (CUTS[2], ref['ref_sig'])]
    for name, r in pairs:
        if name not in tpu_out: continue
        a = tpu_out[name]
        if a.shape != r.shape:
            print(f'{name}: РАЗМЕРЫ ОТЛИЧАЮТСЯ tpu={a.shape} ref={r.shape}'); continue
        err = np.abs(a - r)
        corr = float(np.corrcoef(a.ravel(), r.ravel())[0, 1]) if a.std() > 0 else float('nan')
        print(f'{name}: max|err|={err.max():.4f} mean|err|={err.mean():.5f} corr={corr:.4f}')
print('Сопоставляйте детекции только с эталоном для этого же изображения и source SHA.')

In [ ]:
# 8. Систематический замер времени обработки на LinQ TPU (обязательный пункт кейса:
# 'Инференс на LinQ TPU с измерением времени обработки'). Один и тот же входной тензор
# отправляется N раз; измеряется ПОЛНЫЙ оборот: сеть + брокер + очередь + TPU + ответ.
import json as _json
import pika as _pika

RUNS = 30  # можно увеличить для более устойчивых перцентилей
latencies = []
for k in range(RUNS):
    t0 = time.monotonic()
    tid = uuid.uuid4().hex
    rq = f'notebook_bench_{tid[:8]}'
    conn2 = _pika.BlockingConnection(_pika.ConnectionParameters(
        RABBITMQ_HOST, AMQP_PORT, credentials=_pika.PlainCredentials(*CREDENTIALS)))
    ch2 = conn2.channel()
    ch2.queue_declare(queue=rq, auto_delete=True)
    ch2.queue_bind(exchange=EXCHANGE, queue=rq, routing_key=rq)
    ch2.basic_publish(
        exchange=EXCHANGE, routing_key=INPUT_QUEUE,
        body=serialize(PyTask({INPUT_TENSOR_NAME: tensor})),
        properties=_pika.BasicProperties(headers={
            TARGET_URI_ATTR_NAME: f'file://{PROGRAM_PATH}?v=2&raw=False',
            RESPONSE_ROUTING_KEYS_ATTR_NAME: _json.dumps([rq]),
            MESSAGE_ID_ATTR_NAME: tid,
        }))
    got = False
    for method, props, body in ch2.consume(rq, auto_ack=True, inactivity_timeout=1.0):
        if method is None:
            if time.monotonic() - t0 > TIMEOUT_SECONDS:
                raise TimeoutError(f'нет ответа на итерации {k}')
            continue
        ch2.cancel(); got = True; break
    conn2.close()
    if not got:
        raise RuntimeError(f'нет ответа на итерации {k}')
    d2, m2, w2 = deserialize(body)
    if (not d2 or w2.get('status_code') != 0 or props.headers.get(MESSAGE_ID_ATTR_NAME) != tid):
        raise RuntimeError(f'ошибка воркера: {w2.get("error_message")}')
    lat = time.monotonic() - t0
    latencies.append(lat * 1000)
    print('run %2d/%d: %7.1f ms (round-trip)' % (k + 1, RUNS, lat * 1000))

lat = np.asarray(latencies)
report = {
    'program': PROGRAM,
    'program_sha256': __import__('hashlib').sha256(open(PROGRAM,'rb').read()).hexdigest(),
    'runs': RUNS,
    'input_shape': list(tensor.shape),
    'roundtrip_ms': {'p50': float(np.percentile(lat, 50)),
                     'p95': float(np.percentile(lat, 95)),
                     'mean': float(lat.mean()),
                     'min': float(lat.min()),
                     'max': float(lat.max())},
    'note': 'Full round-trip: RabbitMQ + queue + TPU inference + response transfer. Not the TPU kernel time alone.',
}
open('linq-timing.json', 'w').write(json.dumps(report, indent=2))
print('')
print('ИТОГ (полный оборот, мс): p50=%.0f p95=%.0f mean=%.0f min=%.0f max=%.0f' % (
    np.percentile(lat, 50), np.percentile(lat, 95), lat.mean(), lat.min(), lat.max()))
print('отчёт сохранён: linq-timing.json — скачайте его вместе со скриншотом этой ячейки')

In [ ]:
import sys; print(sys.version)